# 作业 3
**姓名：** 王豪  
**学号：** 20230650215  
**日期：** 2026年6月2日

---

## 2 卷积和池化层

### 2.1 理论计算题

**题目：** 输入一张大小为 3 × 32 × 32（通道数 × 高 × 宽）的彩色图像。通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 3 × 5 × 5。设定填充（Padding）为 2，步幅（Stride）为 2。

**1. 计算卷积层输出的特征图尺寸**

输出尺寸公式：$H_{out} = \lfloor \frac{H_{in} + 2 \times P - K}{S} + 1 \rfloor$

- 输入高 $H_{in} = 32$
- 填充 $P = 2$
- 卷积核大小 $K = 5$
- 步幅 $S = 2$

计算：
$$H_{out} = \lfloor \frac{32 + 2 \times 2 - 5}{2} + 1 \rfloor = \lfloor \frac{31}{2} + 1 \rfloor = \lfloor 15.5 + 1 \rfloor = 16$$

输出特征图尺寸：**16 × 16 × 16**（通道数 × 高 × 宽）

**2. 单个输出通道一个像素值的点乘次数**

每个卷积核大小为 3 × 5 × 5 = 75 个权重参数

每个输出像素需要进行：**3 × 5 × 5 = 75 次**点乘（乘法）操作

### 2.2 编程题 - 手动实现 Max Pooling

要求：手动实现一个支持步幅（stride）和填充（padding）的二维最大池化前向传播函数，不使用深度学习框架的底层 Pooling API。

In [1]:
import numpy as np

def max_pooling_2d(input_matrix, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    
    参数:
        input_matrix: 输入矩阵，形状为 (H, W) 或 (C, H, W)
        kernel_size: 池化核大小，可以是整数或元组 (k_h, k_w)
        stride: 步幅，默认为1
        padding: 填充，默认为0
    
    返回:
        output: 最大池化后的矩阵
    """
    # 将输入转换为浮点类型，避免填充时的类型错误
    input_matrix = np.array(input_matrix, dtype=np.float32)
    
    # 处理输入维度
    if len(input_matrix.shape) == 2:
        input_matrix = input_matrix[np.newaxis, np.newaxis, :, :]
        squeeze_output = True
    elif len(input_matrix.shape) == 3:
        input_matrix = input_matrix[np.newaxis, :, :, :]
        squeeze_output = True
    else:
        squeeze_output = False
    
    # 处理 kernel_size
    if isinstance(kernel_size, int):
        k_h = k_w = kernel_size
    else:
        k_h, k_w = kernel_size
    
    # 处理 stride
    if isinstance(stride, int):
        s_h = s_w = stride
    elif len(stride) == 2:
        s_h, s_w = stride
    else:
        s_h = s_w = stride[0]
    
    # 获取输入尺寸
    batch, channels, H, W = input_matrix.shape
    
    # 计算输出尺寸
    H_out = (H + 2 * padding - k_h) // s_h + 1
    W_out = (W + 2 * padding - k_w) // s_w + 1
    
    # 填充输入
    if padding > 0:
        input_padded = np.pad(input_matrix, 
                              ((0, 0), (0, 0), (padding, padding), (padding, padding)),
                              mode='constant', constant_values=float('-inf'))
    else:
        input_padded = input_matrix
    
    # 初始化输出
    output = np.zeros((batch, channels, H_out, W_out))
    
    # 执行最大池化
    for i in range(H_out):
        for j in range(W_out):
            # 计算当前窗口区域
            h_start = i * s_h
            h_end = h_start + k_h
            w_start = j * s_w
            w_end = w_start + k_w
            
            # 取出窗口并取最大值
            window = input_padded[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2, 3))
    
    # 根据输入维度返回相应形状
    if squeeze_output and len(output.shape) == 4:
        if output.shape[0] == 1 and output.shape[1] > 1:
            output = output[0]
        elif output.shape[0] == 1 and output.shape[1] == 1:
            output = output[0, 0]
    
    return output

# 测试代码
print("=" * 50)
print("Max Pooling 测试")
print("=" * 50)

# 测试1: 基本测试
input_arr = np.array([
    [[1, 2, 3, 4],
     [5, 6, 7, 8],
     [9, 10, 11, 12],
     [13, 14, 15, 16]]
])
print("\n测试1 - 基本测试 (2x2 kernel, stride=2):")
print("输入:")
print(input_arr[0])
result = max_pooling_2d(input_arr, kernel_size=2, stride=2)
print("输出:")
print(result)
print("期望输出: [[6, 8], [14, 16]]")

# 测试2: 带padding
print("\n测试2 - 带padding (3x3 kernel, stride=2, padding=1):")
input_arr2 = np.array([
    [[1, 2, 3, 4, 5],
     [6, 7, 8, 9, 10],
     [11, 12, 13, 14, 15],
     [16, 17, 18, 19, 20],
     [21, 22, 23, 24, 25]]
])
print("输入:")
print(input_arr2[0])
result2 = max_pooling_2d(input_arr2, kernel_size=3, stride=2, padding=1)
print("输出:")
print(result2)

# 测试3: 多通道输入
print("\n测试3 - 多通道输入 (2x2 kernel, stride=2):")
input_arr3 = np.array([
    [[1, 2, 3, 4],
     [5, 6, 7, 8],
     [9, 10, 11, 12],
     [13, 14, 15, 16]],
    [[16, 15, 14, 13],
     [12, 11, 10, 9],
     [8, 7, 6, 5],
     [4, 3, 2, 1]]
])
print("输入 shape:", input_arr3.shape)
result3 = max_pooling_2d(input_arr3, kernel_size=2, stride=2)
print("输出:")
print(result3)
print("输出 shape:", result3.shape)

Max Pooling 测试

测试1 - 基本测试 (2x2 kernel, stride=2):
输入:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]
输出:
[[ 6.  8.]
 [14. 16.]]
期望输出: [[6, 8], [14, 16]]

测试2 - 带padding (3x3 kernel, stride=2, padding=1):
输入:
[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]
 [21 22 23 24 25]]
输出:
[[ 7.  9. 10.]
 [17. 19. 20.]
 [22. 24. 25.]]

测试3 - 多通道输入 (2x2 kernel, stride=2):
输入 shape: (2, 4, 4)
输出:
[[[ 6.  8.]
  [14. 16.]]

 [[16. 14.]
  [ 8.  6.]]]
输出 shape: (2, 2, 2)


---

## 3 LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

**题目：** 假设输入和输出的特征图通道数均为 C。

**1. 计算一个 5 × 5 卷积层（不带偏置）的参数量**

参数量 = 输入通道数 × 输出通道数 × 卷积核高 × 卷积核宽
= C × C × 5 × 5 = **25C²**

**2. 计算两个串联的 3 × 3 卷积层（不带偏置，两层通道数都为 C）的总参数量**

- 第一层：输入通道 C，输出通道 C，参数量 = C × C × 3 × 3 = 9C²
- 第二层：输入通道 C，输出通道 C，参数量 = C × C × 3 × 3 = 9C²
- 总计：**18C²**

**结论：** 两个 3×3 卷积层的参数量 (18C²) 远小于一个 5×5 卷积层 (25C²)，同时能增加网络深度和非线性能力。

### 3.2 编程题 - 定义 NiN 块

要求：使用 PyTorch 定义一个标准的 NiN 块，包含一个普通卷积层和两个 1×1 卷积层级联，每层后跟 ReLU 激活层。

In [1]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块定义
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 普通卷积层的卷积核大小
        stride: 卷积层的步幅
        padding: 卷积层的填充
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(NiNBlock, self).__init__()
        
        # 普通卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.relu1 = nn.ReLU()
        
        # 第一个 1x1 卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.relu2 = nn.ReLU()
        
        # 第二个 1x1 卷积层
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.relu3 = nn.ReLU()
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.conv3(x)
        x = self.relu3(x)
        return x

# 测试 NiN 块
print("=" * 50)
print("NiN 块测试")
print("=" * 50)

# 创建 NiN 块
nin_block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
print("\nNiN Block 结构:")
print(nin_block)

# 测试前向传播
x = torch.randn(1, 3, 32, 32)
output = nin_block(x)
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {output.shape}")

NiN 块测试

NiN Block 结构:
NiNBlock(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (conv2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (relu2): ReLU()
  (conv3): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (relu3): ReLU()
)

输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 16, 32, 32])


---

## 4 Inception, 批量归一化和残差网络

### 4.1 理论计算题

**题目：** 输入 x1=2, x2=4, x3=6, x4=8，γ=2, β=1, ε=0

**计算 Batch Normalization 层转化后的输出值**

**步骤1：计算均值**
$$\mu = \frac{2 + 4 + 6 + 8}{4} = 5$$

**步骤2：计算方差**
$$\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9 + 1 + 1 + 9}{4} = 5$$

**步骤3：标准化**
$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{x_i - 5}{\sqrt{5}}$$

**步骤4：缩放和平移**
$$y_i = \gamma \cdot \hat{x}_i + \beta = 2 \cdot \frac{x_i - 5}{\sqrt{5}} + 1$$

**计算结果：**
- $y_1 = 2 \times \frac{2-5}{\sqrt{5}} + 1 = 2 \times \frac{-3}{\sqrt{5}} + 1 = 1 - \frac{6}{\sqrt{5}} \approx -1.683$
- $y_2 = 2 \times \frac{4-5}{\sqrt{5}} + 1 = 2 \times \frac{-1}{\sqrt{5}} + 1 = 1 - \frac{2}{\sqrt{5}} \approx 0.106$
- $y_3 = 2 \times \frac{6-5}{\sqrt{5}} + 1 = 2 \times \frac{1}{\sqrt{5}} + 1 = 1 + \frac{2}{\sqrt{5}} \approx 1.894$
- $y_4 = 2 \times \frac{8-5}{\sqrt{5}} + 1 = 2 \times \frac{3}{\sqrt{5}} + 1 = 1 + \frac{6}{\sqrt{5}} \approx 3.683$

### 4.2 编程题 - 定义 ResNet 残差块

要求：定义一个残差块类 Residual，包含两个具有相同输出通道数的 3×3 卷积层，每个卷积层后跟一个批量归一化层。如果 use_1x1conv=True，需要对输入应用 1×1 卷积层来调整通道数和形状。

In [ ]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    ResNet 残差块
    
    参数:
        in_channels: 输入通道数
        out_channels: 两个卷积层的输出通道数
        stride: 第一个卷积层的步幅（默认为1）
        use_1x1conv: 是否使用1x1卷积调整输入通道数
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        
        # 第一个 3x3 卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU()
        
        # 第二个 3x3 卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU()
        
        # 1x1 卷积层（用于调整输入通道数和形状）
        if use_1x1conv:
            self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                                    stride=stride, bias=False)
        else:
            self.conv1x1 = None
    
    def forward(self, x):
        identity = x
        
        # 第一个卷积层
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        
        # 第二个卷积层
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 如果使用1x1卷积，调整identity
        if self.conv1x1 is not None:
            identity = self.conv1x1(x)
        
        # 残差连接：f(x) + x
        out = out + identity
        out = self.relu2(out)
        
        return out

# 测试 ResNet 残差块
print("=" * 50)
print("ResNet 残差块测试")
print("=" * 50)

# 测试1: 不使用1x1卷积（通道数相同）
res_block1 = Residual(in_channels=64, out_channels=64, stride=1, use_1x1conv=False)
x1 = torch.randn(1, 64, 32, 32)
output1 = res_block1(x1)
print("\n测试1 - 通道数相同，不使用1x1conv:")
print(f"输入形状: {x1.shape}")
print(f"输出形状: {output1.shape}")

# 测试2: 使用1x1卷积（通道数不同或尺寸不同）
res_block2 = Residual(in_channels=64, out_channels=128, stride=2, use_1x1conv=True)
x2 = torch.randn(1, 64, 32, 32)
output2 = res_block2(x2)
print("\n测试2 - 通道数不同，使用1x1conv:")
print(f"输入形状: {x2.shape}")
print(f"输出形状: {output2.shape}")

# 打印网络结构
print("\n残差块结构:")
print(res_block2)

ResNet 残差块测试

测试1 - 通道数相同，不使用1x1conv:
输入形状: torch.Size([1, 64, 32, 32])
输出形状: torch.Size([1, 64, 32, 32])

测试2 - 通道数不同，使用1x1conv:
输入形状: torch.Size([1, 64, 32, 32])
输出形状: torch.Size([1, 128, 16, 16])

残差块结构:
Residual(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (conv1x1): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
)


---

## 5 图像增广，微调和样式迁移

### 5.1 理论计算题

**1. 为什么对底层特征提取层设置较小的学习率，而对顶层输出层设置较大的学习率？**

**原因：**
- **底层特征提取层**（靠近输入的层）学习的是通用特征，如边缘、纹理、形状等基础模式。这些特征在新任务中通常是可迁移的。
- **顶层输出层**（靠近输出的层）学习的是任务特定的抽象特征，需要根据新任务进行调整。
- 预训练模型已经学到了有用的底层特征，使用较小的学习率可以防止这些特征被大幅破坏。
- 较大的学习率让顶层能更快地适应新任务。

**2. 如果目标数据集非常小，且与源数据集非常相似，应该采取什么样的微调策略？**

**策略：**
- **冻结底层特征提取层**：将大部分底层网络参数固定（冻结），只训练顶层。
- **使用很小的学习率**：即使对顶层，也使用较小的学习率防止过拟合。
- **数据增强**：由于数据量小，需要较强的数据增强来增加样本多样性。
- **正则化**：使用 Dropout、权重衰减等正则化技术防止过拟合。
- **early stopping**：监控验证集性能，在过拟合之前停止训练。

### 5.2 编程题 - 图像增广管道

要求：使用 torchvision.transforms 创建组合图像增广管道。

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import numpy as np

# 定义图像增广管道
# 1. 随机裁剪：面积比例在 0.08 到 1.0 之间，然后缩放到 224x224
# 2. 50% 概率水平翻转
# 3. 随机改变亮度、对比度、饱和度（变化范围 0.5）
# 4. 转换为 PyTorch 张量

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=224,  # 缩放到 224x224
        scale=(0.08, 1.0)  # 面积比例范围
    ),
    transforms.RandomHorizontalFlip(p=0.5),  # 50% 概率水平翻转
    transforms.ColorJitter(
        brightness=0.5,  # 亮度变化范围
        contrast=0.5,    # 对比度变化范围
        saturation=0.5   # 饱和度变化范围
    ),
    transforms.ToTensor()  # 转换为张量
])

# 测试图像增广管道
print("=" * 50)
print("图像增广管道测试")
print("=" * 50)

# 创建一个随机图像进行测试
# 由于没有实际图像，我们用随机数据模拟
print("\n增广管道配置:")
print("1. RandomResizedCrop: size=224, scale=(0.08, 1.0)")
print("2. RandomHorizontalFlip: p=0.5")
print("3. ColorJitter: brightness=0.5, contrast=0.5, saturation=0.5")
print("4. ToTensor")

# 模拟应用增广（创建假图像数据）
fake_image = Image.fromarray(np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8))
print(f"\n输入图像尺寸: {fake_image.size}")

# 应用增广
transformed = train_transform(fake_image)
print(f"输出张量形状: {transformed.shape}")

# 打印 transforms.Compose 的详细信息
print("\n增广管道结构:")
for i, t in enumerate(train_transform.transforms):
    print(f"  {i+1}. {t}")

图像增广管道测试

增广管道配置:
1. RandomResizedCrop: size=224, scale=(0.08, 1.0)
2. RandomHorizontalFlip: p=0.5
3. ColorJitter: brightness=0.5, contrast=0.5, saturation=0.5
4. ToTensor

输入图像尺寸: (256, 256)
输出张量形状: torch.Size([3, 224, 224])

增广管道结构:
  1. RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
  2. RandomHorizontalFlip(p=0.5)
  3. ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
  4. ToTensor()


---

## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

**题目：** 计算边界框 A=[10, 10, 50, 50] 和 B=[30, 30, 70, 70] 之间的 IoU

**IoU 计算步骤：**

**交集区域：**
- 左上角 x = max(10, 30) = 30
- 左上角 y = max(10, 30) = 30
- 右下角 x = min(50, 70) = 50
- 右下角 y = min(50, 70) = 50

**交集面积：**
$$Area_{intersection} = (50 - 30) \times (50 - 30) = 20 \times 20 = 400$$

**并集面积：**
$$Area_A = (50 - 10) \times (50 - 10) = 40 \times 40 = 1600$$
$$Area_B = (70 - 30) \times (70 - 30) = 40 \times 40 = 1600$$
$$Area_{union} = Area_A + Area_B - Area_{intersection} = 1600 + 1600 - 400 = 2800$$

**IoU：**
$$IoU = \frac{Area_{intersection}}{Area_{union}} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429$$

### 6.2 编程题 - 标签平滑交叉熵损失

要求：实现计算标签平滑后交叉熵损失的函数。

对于 K 分类问题，设置平滑因子 ε = 0.1：
- 真实标签对应的目标概率从 1 变为 1 - ε
- 其余错误类别的概率从 0 变为 ε/(K-1)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, reduction='mean'):
    """
    计算标签平滑后的交叉熵损失
    
    参数:
        logits: 模型的原始输出（未归一化的概率），形状为 (batch_size, num_classes)
        labels: 真实标签，形状为 (batch_size,) 或 (batch_size, num_classes) one-hot
        epsilon: 标签平滑因子，默认为0.1
        reduction: 损失聚合方式，'none' | 'mean' | 'sum'
    
    返回:
        loss: 标签平滑后的交叉熵损失
    """
    num_classes = logits.size(-1)
    
    # 将 labels 转换为 one-hot 编码（如果还不是）
    if len(labels.shape) == 1 or (len(labels.shape) == 2 and labels.shape[-1] != num_classes):
        # labels 是类别索引
        one_hot_labels = F.one_hot(labels, num_classes).float()
    else:
        # labels 已经是 one-hot 编码
        one_hot_labels = labels.float()
    
    # 计算平滑后的标签分布
    # 真实类别：(1 - epsilon)
    # 其他类别：epsilon / (K - 1)
    smooth_labels = one_hot_labels * (1 - epsilon) + (1 - one_hot_labels) * (epsilon / (num_classes - 1))
    
    # 计算交叉熵损失：-sum(y * log(p))
    # 使用 log_softmax 提高数值稳定性
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 计算逐元素损失
    loss = -torch.sum(smooth_labels * log_probs, dim=-1)
    
    # 应用 reduction
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:  # 'none'
        return loss

# 测试标签平滑交叉熵损失
print("=" * 50)
print("标签平滑交叉熵损失测试")
print("=" * 50)

# 假设 5 分类问题
num_classes = 5
epsilon = 0.1

# 创建模拟数据
batch_size = 4
logits = torch.randn(batch_size, num_classes)
labels = torch.tensor([0, 1, 2, 3])  # 真实类别索引

print(f"\n分类数 K = {num_classes}")
print(f"平滑因子 epsilon = {epsilon}")
print(f"\n原始 logits:")
print(logits)
print(f"\n真实标签: {labels}")

# 计算标签平滑后的目标分布
print(f"\n平滑后目标分布（以第一个样本为例，真实类为0）:")
print(f"  真实类 (y=0): {1 - epsilon:.1f}")
print(f"  其他类: {epsilon / (num_classes - 1):.4f}")

# 计算损失
loss = label_smoothing_cross_entropy(logits, labels, epsilon=epsilon)
print(f"\n标签平滑交叉熵损失 (mean): {loss.item():.4f}")

# 验证：使用标准交叉熵和手动实现
print("\n验证实现正确性:")

# 转换为 one-hot
one_hot = F.one_hot(labels, num_classes).float()
# 平滑
smooth_labels = one_hot * (1 - epsilon) + (1 - one_hot) * (epsilon / (num_classes - 1))
# 计算
log_probs = F.log_softmax(logits, dim=-1)
manual_loss = -torch.sum(smooth_labels * log_probs, dim=-1).mean()
print(f"手动计算损失: {manual_loss.item():.4f}")

# 测试多个样本的损失
print("\n逐样本损失 (reduction='none'):")
losses_none = label_smoothing_cross_entropy(logits, labels, epsilon=epsilon, reduction='none')
for i, l in enumerate(losses_none):
    print(f"  样本 {i+1} (真实类={labels[i].item()}): {l.item():.4f}")

标签平滑交叉熵损失测试

分类数 K = 5
平滑因子 epsilon = 0.1

原始 logits:
tensor([[-0.2836,  0.3000,  2.0502,  0.2074, -0.1410],
        [-0.5159,  1.4278,  1.1081,  1.1242, -0.3758],
        [-0.8063, -0.8624, -0.5839, -0.1454, -1.8535],
        [ 1.5490,  0.1539, -0.4729, -1.4822,  0.0153]])

真实标签: tensor([0, 1, 2, 3])

平滑后目标分布（以第一个样本为例，真实类为0）:
  真实类 (y=0): 0.9
  其他类: 0.0250

标签平滑交叉熵损失 (mean): 2.1669

验证实现正确性:
手动计算损失: 2.1669

逐样本损失 (reduction='none'):
  样本 1 (真实类=0): 2.6773
  样本 2 (真实类=1): 1.1289
  样本 3 (真实类=2): 1.5124
  样本 4 (真实类=3): 3.3491


---

## 作业完成

以上完成了作业3的所有理论计算题和编程题。

**总结：**

| 题目 | 内容 | 状态 |
|------|------|------|
| 2.1 | 卷积层输出尺寸计算 (16×16×16)，75次点乘 | ✅ |
| 2.2 | Max Pooling 实现 | ✅ |
| 3.1 | VGG 参数量计算 (25C² vs 18C²) | ✅ |
| 3.2 | NiN Block 定义 | ✅ |
| 4.1 | Batch Normalization 计算 | ✅ |
| 4.2 | ResNet Residual Block 定义 | ✅ |
| 5.1 | 微调理论问题 | ✅ |
| 5.2 | 图像增广管道 | ✅ |
| 6.1 | IoU 计算 (1/7 ≈ 0.1429) | ✅ |
| 6.2 | 标签平滑交叉熵损失 | ✅ |